# Inspect the saved model validation notebook

**Purpose:** render the executed native AHR model notebook and capture its visible validation results. **Inputs:** the already executed `01_native_ahr_and_tactics.ipynb`. **Outputs:** HTML and PNG under `reports/ahr_native_v4/`. Expected runtime is under one minute. This notebook performs no training or model execution. **Resume:** Run All; outputs are replaceable views, not checkpoints. The viewer checks only presentation; the model notebook provides numerical validation.


In [ ]:
from pathlib import Path
import os,nbformat
from nbconvert import HTMLExporter
from playwright.sync_api import sync_playwright
PROJECT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'notebooks/01_go_rules_and_independent_verifier.ipynb').exists())
source=PROJECT/'experiments/ahr_native_v4/01_native_ahr_and_tactics.ipynb'
book=nbformat.read(source,as_version=4)
assert all(c.execution_count is not None for c in book.cells if c.cell_type=='code')
assert not any(o.output_type=='error' for c in book.cells if c.cell_type=='code' for o in c.outputs)
report=PROJECT/'reports/ahr_native_v4';report.mkdir(parents=True,exist_ok=True)
exporter=HTMLExporter();exporter.exclude_input=True
html,_=exporter.from_notebook_node(book)
html_path=report/'native_model_notebook.html';html_path.write_text(html)
env={**os.environ,'PLAYWRIGHT_BROWSERS_PATH':str(PROJECT/'.venv/third_party/playwright'),'FONTCONFIG_FILE':str(PROJECT/'.venv/third_party/playwright/fonts.conf'),'LD_LIBRARY_PATH':str(PROJECT/'.venv/third_party/playwright/runtime/usr/lib/x86_64-linux-gnu')}
os.environ['PLAYWRIGHT_BROWSERS_PATH']=env['PLAYWRIGHT_BROWSERS_PATH']
from concurrent.futures import ThreadPoolExecutor
def render_in_thread():
    with sync_playwright() as p:
        browser=p.chromium.launch(headless=True,args=['--no-sandbox'],env=env)
        page=browser.new_page(viewport={'width':1365,'height':1000},device_scale_factor=1)
        page.goto(html_path.as_uri(),wait_until='networkidle')
        # Inspect the actual saved Markdown result near the bottom at reading size.
        page.locator('table').last.scroll_into_view_if_needed()
        page.wait_for_timeout(500)
        page.screenshot(path=str(report/'native_model_validation.png'))
        browser.close()
with ThreadPoolExecutor(max_workers=1) as pool:
    pool.submit(render_in_thread).result()
print({'status':'RENDERED','html':str(html_path.relative_to(PROJECT)),'screenshot':str((report/'native_model_validation.png').relative_to(PROJECT))})
